# Quantum Compass Lab: Simulating Radical-Pair Magnetoreception Under Decoherence
### Qiskit Fall Fest 2026 Research Notebook

This notebook demonstrates the complete scientific simulation workflow:
1. **Spin Hamiltonian Construction:** Electron Zeeman + Anisotropic Hyperfine.
2. **State Preparation:** Two-electron singlet $|S\rangle$ and unpolarized nuclear ensemble $\rho_N = \mathbb{I}/2$.
3. **Angular Response:** Sweeping geomagnetic field orientation $\theta$.
4. **Decoherence Analysis:** Quantifying contrast degradation under spin dephasing ($T_2$).
5. **Qiskit Circuit Simulation:** Lie-Trotter Hamiltonian evolution and cross-validation against SciPy reference.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from src.spin_model import SpinModelConfig
from src.initial_states import get_initial_density_matrix, get_pure_singlet_nuclear_states
from src.hamiltonian import build_total_hamiltonian, build_zeeman_hamiltonian, build_hyperfine_hamiltonian
from src.reaction_kinetics import calculate_yields_exact
from src.evolution import simulate_time_trajectory
from src.experiment_runner import run_angular_sweep
from src.qiskit_backend import create_singlet_prep_circuit, create_trotter_evolution_circuit, simulate_qiskit_evolution
from src.validation import ValidationSuite
from src.plotting import plot_angular_response, plot_decoherence_comparison

print("Quantum Compass Lab modules successfully imported!")

## 1. Radical-Pair Hamiltonian & Eigenvalues
We model an explicit 3-spin system ($S_1, S_2, I$) in an 8-dimensional Hilbert space.

In [ ]:
cfg = SpinModelConfig(
    B_magnitude=50.0,       # Earth field in uT
    theta=0.785,            # ~45 degrees
    phi=0.0,
    A_xx=5.0, A_yy=5.0, A_zz=20.0,  # Axial hyperfine tensor (rad/us)
    k_S=1.0, k_T=1.0        # Recombination rates (1/us)
)

H_tot = build_total_hamiltonian(cfg)
eigs = np.sort(np.linalg.eigvalsh(H_tot))

print(f"Zeeman Frequency omega_0: {cfg.zeeman_omega:.4f} rad/us ({cfg.zeeman_omega / (2*np.pi):.4f} MHz)")
print("Eigenvalues (rad/us):", np.round(eigs, 3))

## 2. Angular Sensitivity Sweep (Earth Field = 50 µT)

In [ ]:
sweep_res = run_angular_sweep(cfg, num_angles=45)
m = sweep_res.metrics

print(f"Compass Contrast C: {m.contrast * 100:.2f}%")
print(f"Absolute Anisotropy Delta_Phi_S: {m.absolute_anisotropy:.4f}")
print(f"Minimum Yield: {m.phi_s_min:.4f} at {m.theta_min_deg:.0f} deg")
print(f"Maximum Yield: {m.phi_s_max:.4f} at {m.theta_max_deg:.0f} deg")

fig = plot_angular_response(
    np.array(sweep_res.thetas_deg),
    np.array(sweep_res.singlet_yields),
    sweep_res.metrics
)
plt.show()

## 3. Decoherence: Decay of Compass Contrast
Sweep spin coherence time $T_2$ and inspect suppression of directional sensitivity.

In [ ]:
gammas = [0.0, 0.5, 2.0, 5.0, 15.0]
labels = ["Ideal", "T2=2.0us", "T2=0.5us", "T2=0.2us", "T2=0.07us"]
yields_dict = {}

for g, lbl in zip(gammas, labels):
    cfg_noisy = SpinModelConfig(B_magnitude=50.0, A_xx=5.0, A_yy=5.0, A_zz=20.0, gamma_dephasing=g)
    res_n = run_angular_sweep(cfg_noisy, num_angles=37)
    yields_dict[lbl] = np.array(res_n.singlet_yields)

fig_decoh = plot_decoherence_comparison(np.array(res_n.thetas_deg), yields_dict)
plt.show()

## 4. Qiskit Circuit Construction & Cross-Validation
Synthesize the Lie-Trotter evolution circuit in Qiskit and compare singlet probability against SciPy exact matrix exponential.

In [ ]:
qc_prep = create_singlet_prep_circuit(nuclear_state=0)
print("Singlet State Preparation Circuit:")
print(qc_prep.draw(output='text'))

t_eval = 0.5 # us
ps_qiskit = simulate_qiskit_evolution(cfg, t_eval, num_trotter_steps=20)
print(f"Qiskit Trotter Singlet Probability at t={t_eval} us: {ps_qiskit:.6f}")

## 5. Automated Scientific Validation Suite
Execute the 12 rigorous automated sanity checks.

In [ ]:
suite = ValidationSuite()
results = suite.run_all()

df_val = pd.DataFrame([{
    'Test': r.name,
    'Passed': r.passed,
    'Metric': f"{r.numerical_metric:.2e}",
    'Tolerance': f"{r.tolerance:.2e}",
    'Details': r.details
} for r in results])

display(df_val)